# 🛡️ Proyecto Pastor Guardián: Detector de Zorros

Este notebook documenta y ejecuta todo el ciclo de vida de desarrollo de nuestro modelo de visión computacional para la protección de ovejas contra el ataque de zorros.

El flujo consta de tres fases principales:
1. **Preparación y Consolidadación de Datos:** Unión de múltiples datasets de zorros en la estructura estándar de YOLO.
2. **Entrenamiento:** Configuración y entrenamiento de un detector de objetos YOLOv11.
3. **Validación:** Evaluación de la precisión del modelo.

## Paso 1: Configuración del Entorno y Verificación de Hardware
Primero, importamos las librerías necesarias y comprobamos si tenemos una GPU CUDA disponible para acelerar el entrenamiento.

In [1]:
import os
import shutil
import glob
import torch
from ultralytics import YOLO

# Comprobar hardware
device = "0" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo seleccionado para el entrenamiento: {device.upper()}")
if device == "0":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("ADVERTENCIA: No se detectó GPU. El entrenamiento en CPU será considerablemente lento.")

Dispositivo seleccionado para el entrenamiento: 0
GPU detectada: NVIDIA GeForce RTX 4060 Laptop GPU


## Paso 2: Consolidación del Dataset YOLO
Unificaremos los datasets de entrada en una única estructura estructurada con subcarpetas `images` y `labels` para `train`, `valid` y `test`.
Añadimos además un prefijo al nombre de cada archivo para evitar colisiones si dos datasets contienen imágenes con el mismo nombre.

In [2]:
workspace = r"d:\SOLEDAD"
dest_root = os.path.join(workspace, "dataset_yolo_consolidado")

datasets = [
    {
        "name": "Sorros-yolo.v3i.yolov11",
        "has_labels": True,
        "subpaths": {
            "train": {"images": os.path.join("train", "images"), "labels": os.path.join("train", "labels")},
            "valid": {"images": os.path.join("valid", "images"), "labels": os.path.join("valid", "labels")},
            "test": {"images": os.path.join("test", "images"), "labels": os.path.join("test", "labels")}
        }
    },
    {
        "name": "Zorros.v2-zorro.yolov11",
        "has_labels": True,
        "subpaths": {
            "train": {"images": os.path.join("train", "images"), "labels": os.path.join("train", "labels")},
            "valid": {"images": os.path.join("valid", "images"), "labels": os.path.join("valid", "labels")},
            "test": {"images": os.path.join("test", "images"), "labels": os.path.join("test", "labels")}
        }
    },
    {
        "name": "fox.v2i.folder",
        "has_labels": False,
        "subpaths": {
            "train": {"images": os.path.join("train", "fox")},
            "valid": {"images": os.path.join("valid", "fox")},
            "test": {"images": os.path.join("test", "fox")}
        }
    }
]

image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.JPG", "*.JPEG", "*.PNG"]

# Inicializar carpetas de destino
for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(dest_root, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(dest_root, split, "labels"), exist_ok=True)

print("Iniciando la consolidación...")
for dataset in datasets:
    ds_name = dataset["name"]
    has_labels = dataset["has_labels"]
    for split, paths in dataset["subpaths"].items():
        img_src = os.path.join(workspace, ds_name, paths["images"])
        img_dest_dir = os.path.join(dest_root, split, "images")
        lbl_dest_dir = os.path.join(dest_root, split, "labels")
        if not os.path.exists(img_src): continue
        
        images = []
        for ext in image_extensions:
            images.extend(glob.glob(os.path.join(img_src, ext)))
            
        prefix = ds_name.replace(".", "_").replace("-", "_")
        for img_path in images:
            filename = os.path.basename(img_path)
            base_name, _ = os.path.splitext(filename)
            new_img_name = f"{prefix}_{filename}"
            
            # Copiar imagen
            shutil.copy2(img_path, os.path.join(img_dest_dir, new_img_name))
            
            # Copiar o generar etiqueta correspondiente
            if has_labels:
                lbl_src = os.path.join(workspace, ds_name, paths["labels"])
                lbl_file = os.path.join(lbl_src, f"{base_name}.txt")
                if os.path.exists(lbl_file):
                    shutil.copy2(lbl_file, os.path.join(lbl_dest_dir, f"{prefix}_{base_name}.txt"))
            else:
                # Crear archivo vacío para imágenes negativas/background
                with open(os.path.join(lbl_dest_dir, f"{prefix}_{base_name}.txt"), "w") as f:
                    pass
print("¡Consolidación finalizada!")

Iniciando la consolidación...
¡Consolidación finalizada!


## Paso 3: Configurar el archivo yaml
Usamos `yolo11n.pt` como punto de partida para entrenar sobre las clases de zorros.

In [3]:
# Crear data.yaml
yaml_content = f"""train: {os.path.join(dest_root, 'train', 'images')}
val: {os.path.join(dest_root, 'valid', 'images')}
test: {os.path.join(dest_root, 'test', 'images')}

nc: 1
names:
  0: zorro
"""
data_yaml = os.path.join(dest_root, "data.yaml")
with open(data_yaml, "w") as f:
    f.write(yaml_content)



 ## y entrenar el modelo

In [4]:
# Inicializar YOLOv11
model = YOLO("yolo11n.pt")

# Iniciar Entrenamiento
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project=os.path.join(workspace, "runs"),
    name="detector_zorros"
)

Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\SOLEDAD\dataset_yolo_consolidado\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detector_zorros-5, nbs=64, nms=False, opset=None, optimize=False, 

## Paso 4: Validación del Modelo
Una vez terminado el entrenamiento, medimos el desempeño del modelo sobre el conjunto de test/validación.

In [ ]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image

# Validar usando el mejor modelo obtenido
best_model_path = os.path.join(workspace, "runs", "detector_zorros", "weights", "best.pt")
run_dir = os.path.join(workspace, "runs", "detector_zorros")

if os.path.exists(best_model_path):
    # Cargar el modelo
    best_model = YOLO(best_model_path)
    
    # Ejecutar validación
    print("Iniciando validación...")
    metrics = best_model.val()
    
    # 1. Mostrar métricas generales
    print("\n================ METRICAS DE VALIDACION ================")
    print(f"mAP50-95 (Precisión Media Promedio): {metrics.box.map:.4f}")
    print(f"mAP50:                             {metrics.box.map50:.4f}")
    print(f"Precisión media (Precision):       {metrics.box.mp:.4f}")
    print(f"Sensibilidad media (Recall):       {metrics.box.mr:.4f}")
    
    # Calcular F1-Score promedio
    if (metrics.box.mp + metrics.box.mr) > 0:
        f1_score = 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr)
        print(f"F1-Score promedio:                 {f1_score:.4f}")
    else:
        print("F1-Score promedio:                 0.0000")
    print("========================================================\n")
    
    # 2. Desglose de métricas por cada clase detectada (Corregido)
    print("--- Métricas por Clase ---")
    names = best_model.names
    # Si 'classes' no está en metrics, usamos un rango del tamaño de los resultados
    classes_eval = getattr(metrics, 'classes', list(range(len(metrics.box.p))))
    
    for i, c in enumerate(classes_eval):
        class_name = names.get(int(c), f"Clase {c}")
        class_p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
        class_r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
        class_ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
        class_ap = metrics.box.ap[i] if i < len(metrics.box.ap) else 0.0
        print(f"Clase '{class_name}': Precisión = {class_p:.4f}, Recall = {class_r:.4f}, AP50 = {class_ap50:.4f}, AP50-95 = {class_ap:.4f}")
    print("========================================================\n")
    
    # 3. Cargar y mostrar los gráficos de rendimiento generados
    plots = [
        {"titulo": "Diagrama de Aprendizaje (Pérdidas y Métricas)", "ruta": os.path.join(run_dir, "results.png")},
        {"titulo": "Matriz de Confusión", "ruta": os.path.join(metrics.save_dir, "confusion_matrix.png")},
        {"titulo": "Matriz de Confusión Normalizada", "ruta": os.path.join(metrics.save_dir, "confusion_matrix_normalized.png")},
        {"titulo": "Curva de Precisión-Recall (PR)", "ruta": os.path.join(metrics.save_dir, "PR_curve.png")},
        {"titulo": "Curva F1", "ruta": os.path.join(metrics.save_dir, "F1_curve.png")}
    ]
    
    for plot in plots:
        if os.path.exists(plot["ruta"]):
            print(f"Mostrando gráfico: {plot['titulo']}")
            img = Image.open(plot["ruta"])
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.axis('off')
            plt.title(plot["titulo"], fontsize=14, pad=10)
            plt.show()
        else:
            nombre_archivo = os.path.basename(plot["ruta"])
            ruta_alternativa = os.path.join(run_dir, nombre_archivo)
            if os.path.exists(ruta_alternativa):
                print(f"Mostrando gráfico (alternativo): {plot['titulo']}")
                img = Image.open(ruta_alternativa)
                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.axis('off')
                plt.title(plot["titulo"], fontsize=14, pad=10)
                plt.show()
            else:
                print(f"Aviso: No se encontró el gráfico '{nombre_archivo}'")
else:
    print("El modelo entrenado no fue encontrado en la ruta esperada.")


Iniciando validación...
Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 617.5333.5 MB/s, size: 51.1 KB)
val: Scanning D:\SOLEDAD\dataset_yolo_consolidado\valid\labels.cache... 103 images, 51 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 103/103  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 13, len(boxes) = 54. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.7it/s 4.1s0.2s6
                   all        103         54      0.338      0.111      0.116     0.0501
Speed: 2.5ms preprocess, 4.4ms inference, 0.0ms loss, 2.4ms postprocess per im

<Figure size 1000x800 with 1 Axes>

Mostrando gráfico: Matriz de Confusión Normalizada


<Figure size 1000x800 with 1 Axes>

Aviso: No se encontró el gráfico 'PR_curve.png'
Aviso: No se encontró el gráfico 'F1_curve.png'


: 